In [2]:
import torch
import torch.nn as nn
import numpy as np
import pandas as  pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:], df.iloc[:,0], test_size=0.2)

In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [8]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [9]:
X_train_tensor = X_train_tensor.float()
y_train_tensor = y_train_tensor.float()
X_test_tensor = X_test_tensor.float()

In [1]:
from torch.utils.data import DataLoader,Dataset

In [10]:
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features = features
    self.labels = labels
  def __len__(self):
    return len(self.features)

  def __getitem__(self, index) :
    return self.features[index],self.labels[index]


In [13]:
train_dataset  = CustomDataset(X_train_tensor,y_train_tensor)
test_dataset = CustomDataset(X_test_tensor,y_test_tensor)

train_dataset[10]


(tensor([-0.3041,  0.6008, -0.1661, -0.3694,  2.2226,  1.7907,  1.3087,  1.1994,
          2.0694,  1.5996, -0.3497, -0.3782, -0.2262, -0.3375, -0.4522,  0.5956,
          0.1695,  0.0708,  0.1223,  0.0085, -0.1505,  0.8492, -0.0209, -0.2377,
          1.6404,  1.8205,  1.3161,  1.4042,  2.3544,  1.2676]),
 tensor(1.))

In [14]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=True)

In [15]:
class mySimpleRnn(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(num_features,10),
        nn.ReLU(),
        nn.Linear(10,1),
        nn.Sigmoid()
    )

  def forward(self,X):
    out = self.network(X)
    return out

In [18]:
lr = 0.1
epochs = 25

In [19]:
model = mySimpleRnn(X_train_tensor.shape[1])
optimizer = torch.optim.Adam(model.parameters(),lr)
loss_func = nn.BCELoss()

In [20]:
# running the model
for epoch in range(epochs):
  for batch_features,batch_labels in train_loader:
    # forward pass
    y_pred = model(batch_features)
    # calculate loss
    loss = loss_func(y_pred,batch_labels.view(-1,1))
    # zero gradient
    optimizer.zero_grad()
    # backward pass
    loss.backward()
    # parameter update
    optimizer.step()

  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')


Epoch: 1, Loss: 0.04710465297102928
Epoch: 2, Loss: 0.0009701213566586375
Epoch: 3, Loss: 0.004895147867500782
Epoch: 4, Loss: 0.0006230300641618669
Epoch: 5, Loss: 0.05214307829737663
Epoch: 6, Loss: 0.7222792506217957
Epoch: 7, Loss: 0.06015994772315025
Epoch: 8, Loss: 0.3188210129737854
Epoch: 9, Loss: 0.0037260146345943213
Epoch: 10, Loss: 1.2864101336163003e-05
Epoch: 11, Loss: 0.009716845117509365
Epoch: 12, Loss: 0.01409605611115694
Epoch: 13, Loss: 0.0001566540013300255
Epoch: 14, Loss: 0.1339827924966812
Epoch: 15, Loss: 0.12247662991285324
Epoch: 16, Loss: 3.0154265914461575e-05
Epoch: 17, Loss: 0.0014426267007365823
Epoch: 18, Loss: 4.342182819527807e-06
Epoch: 19, Loss: 0.0036406703293323517
Epoch: 20, Loss: 0.030952276661992073
Epoch: 21, Loss: 0.00010731018119258806
Epoch: 22, Loss: 0.00010414963617222384
Epoch: 23, Loss: 0.0002196561254095286
Epoch: 24, Loss: 4.958807267030352e-07
Epoch: 25, Loss: 0.11653278768062592


In [21]:
# Model evaluation using test_loader
model.eval()# Set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
      # Forward pass
      y_pred = model(batch_features)
      y_pred = (y_pred > 0.8).float()  # Convert probabilities to binary predictions

      # Calculate accuracy for the current batch
      batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
      accuracy_list.append(batch_accuracy)

# Calculate overall accuracy
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Accuracy: {overall_accuracy:.4f}')


Accuracy: 0.9627
